In [17]:
%%! 
pip install -r requirements.txt

['Requirement already satisfied: pandas in c:\\program files\\python312\\lib\\site-packages (from -r requirements.txt (line 1)) (3.1.1)',
 'Requirement already satisfied: seaborn in c:\\program files\\python312\\lib\\site-packages (from -r requirements.txt (line 2)) (0.13.2)',
 'Requirement already satisfied: matplotlib in c:\\program files\\python312\\lib\\site-packages (from -r requirements.txt (line 3)) (3.11.2)',
 'Requirement already satisfied: numpy in c:\\program files\\python312\\lib\\site-packages (from -r requirements.txt (line 4)) (2.5.5)',
 'Requirement already satisfied: SQLAlchemy in c:\\program files\\python312\\lib\\site-packages (from -r requirements.txt (line 5)) (2.1.9)',
 'Requirement already satisfied: pyodbc in c:\\program files\\python312\\lib\\site-packages (from -r requirements.txt (line 6)) (5.5.0)',
 'Collecting argparse (from -r requirements.txt (line 7))',
 '  Downloading argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)',
 'Requirement already satisfie

In [16]:
import pyodbc 
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)']


In [4]:
# Pull data 

def readCSV(url):
    return pd.read_csv(url)

# Book cleanup 

def cleanBooks(systembook):
    
    # Remove nulls / Nan
    systembook = systembook.dropna(how="all")

    # Strip the '"" from dates
    systembook["Book checkout"] = systembook["Book checkout"].str.strip('"')
    
    # Convert IDs to int 
    systembook["Customer ID"] = systembook["Customer ID"].astype("Int64")
    systembook["Id"] = systembook["Id"].astype("Int64")
    
    # Convert weeks to days 
    systembook["Days allowed to borrow"] = systembook["Days allowed to borrow"].str.extract(r"(\d+)", expand=False).astype("Int64") * 7 
    
    return systembook

def validateBooks(systembook): 

    # Check for empty book names
    systembook["Valid Rows"] = "Pass"
    systembook["Valid Rows"] = systembook["Valid Rows"].mask(systembook["Books"].isna(), "Fail - invalid book name")    
    
    # Convert to datetime + remove invalid dates 
    for c in ["Book checkout", "Book Returned"]:
        systembook[c] = pd.to_datetime(systembook[c], format="%d/%m/%Y", errors="coerce")

    # Invalid date
    systembook["Valid Rows"] = systembook["Valid Rows"].mask(systembook["Book checkout"].isna() | systembook["Book Returned"].isna(), "Fail - invalid date")

    # Book checkout >= Book Returned
    systembook["Valid Rows"] = systembook["Valid Rows"].mask(systembook["Book checkout"] >= systembook["Book Returned"], "Fail - checkout not before return")

    # The checkout date or return is in the future
    systembook["Valid Rows"] = systembook["Valid Rows"].mask((systembook["Book checkout"] > pd.Timestamp.today()) | (systembook["Book Returned"] > pd.Timestamp.today()), "Fail - future date")

    return systembook

def cleanupErrors(systembook):
    
    systembook = systembook[systembook["Valid Rows"] == "Pass"]
    systembook = systembook.drop(columns=["Valid Rows"])   
    return systembook

# Customer Cleanup 

def cleanAndValidateCustomers(systemCustomers): 

    # Remove empty or NAN rows 
    systemCustomers = systemCustomers.dropna(how="all")

    # Convert IDs to int 
    systemCustomers["Customer ID"] = systemCustomers["Customer ID"].astype("Int64")

    return systemCustomers

# Output 

def outputFiles(systembook,systemCustomers):

    import os

    # Move failed into seperate folder

    invalidRows_systembook = systembook[systembook["Valid Rows"] != "Pass"]

    # Cleanup failed + remove column

    systembook = cleanupErrors(systembook)

    os.makedirs("exportData", exist_ok=True)
    
    systembook.to_csv("exportData/systembook.csv", index=False)
    systemCustomers.to_csv("exportData/systemCustomers.csv", index=False)
    invalidRows_systembook.to_csv("exportData/systembook_invalidRows.csv", index=False)

In [5]:
import pandas as pd

# Pull the data 

systembook = readCSV("data/03_Library Systembook.csv")
systemCustomers = readCSV("data/03_Library SystemCustomers.csv")

# Sort the Books 

systembook = cleanBooks(systembook)
systembook = validateBooks(systembook)

# Sort the customers

systemCustomers = cleanAndValidateCustomers(systemCustomers)

# Output

outputFiles(systembook,systemCustomers)

C:\Users\Admin\AppData\Local\Temp\ipykernel_11136\3916621968.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  systembook["Book checkout"] = systembook["Book checkout"].str.strip('"')
C:\Users\Admin\AppData\Local\Temp\ipykernel_11136\3916621968.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  systembook["Customer ID"] = systembook["Customer ID"].astype("Int64")
C:\Users\Admin\AppData\Local\Temp\ipykernel_11136\3916621968.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice